In [3]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1/BIGFAM")

import famgen.gen_geno as fgg
import famgen.gen_pheno as fgp
from reml_utils import *

# simulation

In [4]:
# GENOTYPE-related Parameters
num_sample = 100
num_variant = 100
prop_commom_rare = -1 # only common variant (0 for only rare)

In [5]:
# PHENOTYPE related parameters
dc_type = "FDC"
beta = {"autosome" : np.random.normal(0, 1, size=num_variant),
        "chrX" : np.random.normal(0, 1, size=num_variant)}

hsqs = {
    "autosome" : 0.5,
    "chrX" : 0,
    "fam" : 0.05,
    "po" : 0.05,
    "sib" : 0.1,
}

In [6]:
# genotype-related add-hoc functions
def make_parents_haps(class_FamGenoSimul, num_sample):
    """Generate parent haplotypes.

    This function generates the haplotypes for the father and mother. The haplotypes
    are stored in a dictionary with the keys "father" and "mother". Each parent has
    two keys, "autosome" and "chrX".

    Args:
        class_FamGenoSimul (class): The FamGenoSimul class instance.
        num_sample (int): The number of samples to generate.

    Returns:
        dict: A dictionary of the generated parent haplotypes.
    """
    dict_parents = {
        "father" : {
            "autosome" : None,
            "chrX" : None},
        "mother" : {
            "autosome" : None,
            "chrX" : None}
    }

    ## MAKE PARENT's HAPS
    for parent in ["father", "mother"]:
        for chr_type in ["autosome", "chrX"]:
            if (chr_type == "chrX") & (parent == "father"):
                dict_parents[parent][chr_type], _ = class_FamGenoSimul.generate_parent_haplotype(num_sample=num_sample, chr_type="haploid")
            else:
                dict_parents[parent][chr_type], _ = class_FamGenoSimul.generate_parent_haplotype(num_sample=num_sample, chr_type="diploid")
    
    return dict_parents

def make_offspring_haps(class_FamGenoSimul, dict_parents, off_type):
    # off_type = "son", "daughter"
    dict_off = {
        "autosome" : None,
        "chrX" : None
    }
    for chr_type in ["autosome", "chrX"]:
        dict_off[chr_type] = class_FamGenoSimul.make_offspring_haplotype(
            haps_mother = dict_parents["mother"][chr_type],
            haps_father = dict_parents["father"][chr_type],
            off_type = off_type,
            chr_type=chr_type
        )
    return dict_off

def make_family_geno(class_FamGenoSimul, num_sample, rel):
    """
    Generate offspring's genotype for the specified relationship type.
    
    Parameters:
    class_FamGenoSimul (class): Class for family simulation.
    num_sample (int): The number of samples to be simulated.
    rel (str): Relationship type. 
        Options: father_son, mother_son, father_daughter, mother_daughter, son_son, son_daughter, daughter_daughter
        
    Returns:
    dict: Dictionary with offspring's genotype for each chromosome type.
    
    """
    parents = ["father", "mother"]
    offs = ["son", "daughter"]
    pos = ["father_son", "mother_son", "father_daughter", "mother_daughter"]
    sibs = ["son_son", "son_daughter", "daughter_daughter"]
    
    dict_parents = make_parents_haps(class_FamGenoSimul, num_sample)
    
    ## MAKE OFFSPRING's HAPS ##
    if rel in ["father_son", "mother_son"]:
        dict_off = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="son")
        
    elif rel in ["father_daughter", "mother_daughter"]:
        dict_off = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="daughter")

    elif rel == "son_son":
        dict_off1 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="son")
        dict_off2 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="son")

    elif rel == "son_daughter":
        dict_off1 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="son")
        dict_off2 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="daughter")

    elif rel == "daughter_daughter":
        dict_off1 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="daughter")
        dict_off2 = make_offspring_haps(class_FamGenoSimul, dict_parents, off_type="daughter")

    else: 
        raise Exception("!!")

    # return dict_parents
    dict_rel = {}
    
    if rel in pos:
        for r in rel.split("_"):
            dict_rel[r] = {}
            for chr_type in ["autosome", "chrX"]:
                if r in parents:
                    dict_rel[r][chr_type] = dict_parents[r][chr_type]
                elif r in offs:
                    dict_rel[r][chr_type] = dict_off[chr_type]
    
    elif rel in sibs:
        for i, r in enumerate(rel.split("_")):
            r_new = r + str(i+1)
            dict_rel[r_new] = {}
            for chr_type in ["autosome", "chrX"]:
                if i == 0:
                    dict_rel[r_new][chr_type] = dict_off1[chr_type]
                else:
                    dict_rel[r_new][chr_type] = dict_off2[chr_type]
            
    return dict_rel

In [7]:
GenGeno = fgg.FamGenoSimul(num_variant, prop_commom_rare)
GenGeno

FamGenoSimul(num_variant=100, prop_common_rare=-1, maf_lim_common=[0.05, 0.95], maf_lim_rare=[0.01, 0.05], num_common=100, num_rare=0)

In [8]:
reltypes = [
    "father_son",
    "brother",
    "mother_daughter",
    "sister"
]

pos = [
    "father_son",
    "mother_daughter",
]

sibs = [
    "brother",
    "sister"
]

males = [
    "father_son",
    "brother",
]

females = [
    "mother_daughter",
    "sister"
]

In [9]:
# make phenotype

df_rel = pd.DataFrame(columns=["eid", "eid_rel", "pheno", "pheno_rel", "relation"])
idx = 100000
geno_r = {"autosome": pd.DataFrame(columns=np.arange(num_variant)),
          "chrX": pd.DataFrame(columns=np.arange(num_variant))}

for reltype in reltypes:
    # convert reltype appropriate for `famgen`
    if reltype == "brother":
        r = "son_son"
    elif reltype == "sister":
        r = "daughter_daughter"
    else:
        r = reltype
    
    # make genotype for given relationship
    rel_pair = make_family_geno(GenGeno, num_sample, r)
    geno_r[r] = rel_pair
    
    # make shared env
    s_fam = np.random.normal(0, 1, num_sample)
    s_relspec = np.random.normal(0, 1, num_sample) # s_po for po, s_sib for sib
    
    # make phenotype
    pheno_r = {}
    comps = ["autosome", "chrX", "fam", "po"] if r in pos else ["autosome", "chrX", "fam", "sib"]
    
    for rr in rel_pair.keys():
        pheno = np.zeros(num_sample)

        haps = rel_pair[rr]
        hsq_sum = 0
        for comp in comps:
            if comp == "autosome":
                pheno_comp = fgp.make_pheno(hsqs[comp], haps[comp], beta[comp])
                
            elif comp == "chrX":
                if rr in males:
                    pheno_comp = fgp.make_pheno(hsqs[comp], haps[comp], beta[comp])
                else:
                    pheno_comp = fgp.make_pheno(hsqs[comp], haps[comp], beta[comp], dc_type=dc_type)
            
            elif comp == "fam":
                pheno_comp = fgp.set_var(hsqs[comp], s_fam)
            
            else:
                pheno_comp = fgp.set_var(hsqs[comp], s_relspec)
            
            pheno += pheno_comp
            hsq_sum += hsqs[comp]
        
        e = np.random.normal(0, 1, num_sample)
        var_e = 1 - hsq_sum
        e = fgp.set_var(var_e, e)
        pheno += e
    
        pheno_r[rr] = pheno
    
    # make rel
    idx_m1 = np.arange(idx+1, idx + num_sample + 1)
    idx_m2 = np.arange(idx + num_sample + 1, idx + 2 * num_sample + 1)
    df_tmp = pd.DataFrame({"eid": idx_m1,
                           "eid_rel": idx_m2,
                           "pheno": pheno_r[list(pheno_r.keys())[0]],
                           "pheno_rel": pheno_r[list(pheno_r.keys())[1]],
                           "relation": reltype})
    idx += 2 * num_sample
    df_rel = pd.concat([df_rel, df_tmp], axis=0).reset_index(drop=True)
    
    # make genotype matrix
    dict_geno = {}
    for chrtype in ["autosome", "chrX"]:
        tmp_geno = pd.DataFrame()
        for ir, rr in enumerate(rel_pair.keys()):
            idx_id = idx_m1 if ir == 0 else idx_m2
            tt = pd.DataFrame(fgp.hap_to_geno(rel_pair[rr][chrtype]), 
                              index=idx_id)
            if (rr in ["father", "son"]) & (chrtype == "chrX"):
                tt = tt * 2
                
            tmp_geno = pd.concat([tmp_geno, tt], axis=0)
        dict_geno[chrtype] = tmp_geno

    for chrtype in ["autosome", "chrX"]:
        geno_r[chrtype] = pd.concat([geno_r[chrtype], dict_geno[chrtype]], axis=0)
        
            

In [10]:
df_pheno = pd.concat([df_rel[["eid", "pheno"]],
                      df_rel[["eid_rel", "pheno_rel"]].rename(columns={
                          "eid_rel": "eid",
                          "pheno_rel": "pheno"
                          })], axis=0).reset_index(drop=True)
df_pheno

,eid,pheno
0,100001,0.215042
1,100002,-0.277481
2,100003,0.643361
3,100004,0.096283
4,100005,-1.332811
...,...,...
795,100796,0.601846
796,100797,0.944064
797,100798,1.966602
798,100799,-3.445037


In [11]:
# make eGRM
df_rels = df_rel[["eid", "eid_rel", "relation"]]
egrm_A = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                      columns=df_pheno["eid"].values,
                      index=df_pheno["eid"].values)

for i in range(df_rels.shape[0]):
    m1, m2, rel_type = df_rels.iloc[i, :]
    egrm_A.loc[m1, m2] = 1/2
    egrm_A.loc[m2, m1] = 1/2

egrm_fam = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                      columns=df_pheno["eid"].values,
                      index=df_pheno["eid"].values)

for i in range(df_rels.shape[0]):
    m1, m2, rel_type = df_rels.iloc[i, :]
    egrm_fam.loc[m1, m2] = 1
    egrm_fam.loc[m2, m1] = 1

egrm_po = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                      columns=df_pheno["eid"].values,
                      index=df_pheno["eid"].values)

for i in range(df_rels.shape[0]):
    m1, m2, rel_type = df_rels.iloc[i, :]
    if rel_type in pos:
        egrm_po.loc[m1, m2] = 1
        egrm_po.loc[m2, m1] = 1
        
egrm_sib = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                      columns=df_pheno["eid"].values,
                      index=df_pheno["eid"].values)

for i in range(df_rels.shape[0]):
    m1, m2, rel_type = df_rels.iloc[i, :]
    if rel_type in sibs:
        egrm_sib.loc[m1, m2] = 1
        egrm_sib.loc[m2, m1] = 1

In [12]:
grm = [egrm_A, egrm_fam, egrm_po, egrm_sib]
h0 = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
pheno = df_pheno["pheno"]

gcta_fs = aiREML(grm, pheno, h0, max_iter=10, verbose = True)

Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-566.3553812630988 0.0001 1.7764561482044736
-2642.275112559984 0.0001 513.7764561482045
-5268.213497376925 0.0001 0.0001
-10350.207345476829 137438953472.0001 0.0001
-21930.644277242624 3.022314549037947e+23 0.0001
-45297.84580126101 1.7538019647970835e+49 0.0001
-91271.69556742565 0.0001 0.0001
nan nan nan


/home/jerrylee/miniconda3/envs/BIGFAM/lib/python3.9/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


# real

In [13]:
df_dor = pd.read_csv(
    "/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/relative_information/relatives.formatted.info",
    sep='\t'
)
df_dor

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,1000094,3653174,65,64,F,F,0.75
1,1,NaN,NaN,1000220,1691267,64,64,F,F,NaN
2,1,NaN,NaN,1000286,1571411,53,70,F,F,NaN
3,1,NaN,NaN,1000295,1045127,60,41,F,F,NaN
4,1,NaN,NaN,1000476,3599303,50,51,F,M,NaN
...,...,...,...,...,...,...,...,...,...,...
81321,3,1C,son-(father-brother)/(father-sister)-daughter,6023723,4863061,62,64,F,M,0.00
81322,3,NaN,NaN,6024211,1209127,53,60,M,F,NaN
81323,3,NaN,NaN,6024384,1854265,62,44,M,M,NaN
81324,3,1C,son-(father-sister)/(father-sister)/(mother-br...,6024486,3148753,58,56,M,M,0.00


In [32]:
# 200 pairs for each DOR
df_under = df_dor.groupby("DOR").sample(300)
df_under

df_id = pd.DataFrame({
    "eid": list(set(df_under["volid"].unique()) | set(df_under["relid"].unique()))
})
df_id

,eid
0,5476353
1,4521987
2,2371589
3,2715654
4,5603333
...,...
1787,3133425
1788,1806326
1789,5898233
1790,2465787


In [33]:
pheno_path = "/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/phenotype"
pheno_fns = os.listdir(pheno_path)

In [34]:
df_res = pd.DataFrame(columns=["pheno", "error", "values"])

for ii, pheno_fn in enumerate(pheno_fns):
    pheno =  pheno_fn.split(".")[0]
    print(f"[{ii+1}/{len(pheno_fns)}] {pheno}")
    
    # phenotype
    df_pheno = pd.read_csv(f"{pheno_path}/{pheno_fn}",
                           sep='\t',)
    df_pheno = pd.merge(df_id, df_pheno[["eid", "pheno"]])
    
    df_rels = df_under[["volid", "relid", "DOR"]]
    df_rels = df_rels[(df_rels["volid"].isin(df_pheno["eid"])) & (df_rels["relid"].isin(df_pheno["eid"]))]
    
    # make eGRM
    egrm_A = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                        columns=df_pheno["eid"].values,
                        index=df_pheno["eid"].values)

    for i in range(df_rels.shape[0]):
        m1, m2, dor = df_rels.iloc[i, :]
        egrm_A.loc[m1, m2] = (1/2)**dor
        egrm_A.loc[m2, m1] = (1/2)**dor

    egrm_d1 = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                        columns=df_pheno["eid"].values,
                        index=df_pheno["eid"].values)

    for i in range(df_rels.shape[0]):
        m1, m2, dor = df_rels.iloc[i, :]
        if dor == 1:
            egrm_d1.loc[m1, m2] = 1
            egrm_d1.loc[m2, m1] = 1

    egrm_d2 = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                        columns=df_pheno["eid"].values,
                        index=df_pheno["eid"].values)

    for i in range(df_rels.shape[0]):
        m1, m2, dor = df_rels.iloc[i, :]
        if dor == 2:
            egrm_d2.loc[m1, m2] = 1
            egrm_d2.loc[m2, m1] = 1

    egrm_d3 = pd.DataFrame(np.eye(df_pheno.shape[0]).astype(int),
                        columns=df_pheno["eid"].values,
                        index=df_pheno["eid"].values)

    for i in range(df_rels.shape[0]):
        m1, m2, dor = df_rels.iloc[i, :]
        if dor == 3:
            egrm_d3.loc[m1, m2] = 1
            egrm_d3.loc[m2, m1] = 1

    try:
        grm = [egrm_A, egrm_d1, egrm_d2, egrm_d3]
        h0 = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
        pheno_val = df_pheno["pheno"]

        gcta_fs = aiREML(grm, pheno_val, h0, max_iter=10, verbose = True)
        df_res.loc[len(df_res)] = [pheno, False, gcta_fs]
    except:
        df_res.loc[len(df_res)] = [pheno, True, None]
    

[1/106] White_blood_cell__leukocyte__count
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-1558.5047400422995 0.0001 2.5260193745654824
-4522.629584149958 0.0001 66.52601937456548
-8407.39999374194 16384.0001 0.0001
-18382.140889564707 0.0001 1073741824.0001
36177.179574026384 0.0001 0.0001
-70997.75197898233 7.875675875025626e+34 9.844594843782034e+34
-142588.03316041947 0.0001 2.7606985387162255e+70
-1393465.7058103352 0.0001 0.0001
-398113.93689369474 0.0001 0.0010765625
-411716.5460954413 0.0007171875 0.00045937500000000004
[2/106] Lymphocyte_count
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
17.46083708049946 0.025663451348531104 0.09879688030270736
19.82381290335786 0.0861478263485311 0.07472656780270737
-921.5247341321639 0.0001 1.2523203178027074
-83819.60303896126 0.0001 0.0001
-167.8655176448915 0.21347890625 0.0001
-70.245909584521

/data/jerrylee/pjt/BIGFAM.v.0.1/BIGFAM/reml_utils.py:170: RuntimeWarning: invalid value encountered in sqrt
  return [np.sqrt(SE), Sinv]


[3/106] Trunk_fat-free_mass
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-835.712022374492 0.0001 0.32081237648110805
-986.8747111748023 0.0001 0.7997498764811081
-1057.0075208080002 0.0001 0.0001
-1244.7523421611954 2.5001 0.7501
-3644.8768701575677 66.5001 0.0001
-8338.094618208119 0.0001 8192.0001
-344733.67365876137 0.0001 0.0001
-31741.623766475357 0.0001 0.0001
-66487.54227047687 8.203829036485028e+32 0.0001
-133581.9963562303 1.2637475000226862e+66 0.0001
[4/106] Whole_body_water_mass
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-825.3712444961074 0.17284541684534538 0.19409706415714242
-843.5784597480824 0.0001 0.44837831415714247
-820.6091295468351 0.1251 0.26087831415714247
-820.0030565423641 0.03622499999999999 0.31025331415714247
-873.3198691291817 1.0831 0.0001
-1302.2111688531086 0.0001 1.7501
-2024.7405292212131 8.0001 0.0001

/home/jerrylee/miniconda3/envs/BIGFAM/lib/python3.9/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


[24/106] Glucose
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-834.4923209813139 0.24228777219163922 0.12733864007699291
-1014.810949812394 0.0001 0.848213640076993
-872.2347803316536 0.0001 0.0001
-877.6738183807606 0.0001 0.1976
-1214.064956262197 3.5001 0.0001
-1589.4635547234545 3.5001 0.0001
-3653.691587677595 0.0001 64.0001
-4997.544240126023 0.0001 0.0001
8436.08813073159 0.0001 131072.0001
-15876.89272840689 0.0001 0.0001
[25/106] High_light_scatter_reticulocyte_percentage
Perform a single iteration of EM-based REML to initiate parameters
LogLike V(G) V(e)
Iterate AI REML until convergence
-384.4660474528887 1.1921032720168678 0.0001
-654.4281287257201 0.6921032720168678 0.0001
-1198.1900261212597 2.692103272016868 1.0001
-3903.8774910892425 0.0001 33.0001
-9459.255289793366 65536.0001 0.0001
-18616.951457380273 65536.0001 0.0001
-39525.6218079402 1.1068046444225738e+20 0.0001
-79851.33200079124 2.1778071

In [39]:
df_res[df_res["error"] == False]

,pheno,error,values
13,Whole_body_fat_mass,False,"[[nan, nan, nan, nan, nan], [nan, nan, nan, na..."
20,Red_blood_cell__erythrocyte__distribution_width,False,"[[1.7337875771562706e+106, 1.3070827984855963e..."
25,Townsend_deprivation_index_at_recruitment,False,"[[nan, nan, nan, nan, nan], [nan, nan, nan, na..."
46,Arm_fat-free_mass__left_,False,"[[nan, nan, nan, nan, nan], [nan, nan, nan, na..."
49,Whole_body_fat-free_mass,False,"[[1.3838256509620206e+88, 0.0001, 4.6705923953..."
54,MET_minutes_per_week_for_moderate_activity,False,"[[0.004235678940827401, 0.0032931351399160145,..."
55,Leg_predicted_mass__left_,False,"[[2.5357771281718143e+130, 0.0001, 0.0001, 0.0..."
63,Nucleated_red_blood_cell_count,False,"[[nan, nan, nan, nan, nan], [nan, nan, nan, na..."
65,Impedance_of_arm__left_,False,"[[6.40982242619366e+141, 4.747903422433448e+14..."
70,Body_fat_percentage,False,"[[nan, nan, nan, nan, nan], [nan, nan, nan, na..."
